In [ ]:
# =============================================================================
# FYP EDGE AI DEPLOYMENT: AUTOENCODER EXPORT PIPELINE
# Dataset: IMS (Run-to-Fail Bearing Data)
# Architecture: Unsupervised ConvLSTM-Autoencoder
# Target Device: Raspberry Pi 4 (TensorFlow Lite)
# Author: Vincent Tay Yong Jun
# =============================================================================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from google.colab import drive, files
import pandas as pd

# =============================================================================
# STEP 1: Environment Setup & Google Drive Mounting
# =============================================================================
print("=== STEP 1: MOUNTING GOOGLE DRIVE ===")
drive.mount('/content/drive')

# =============================================================================
# STEP 2: Signal Preprocessing Functions
# =============================================================================
WINDOW_SIZE = 1024
STRIDE = 1024

def create_sliding_windows(data, window_size, stride):
    """ Segments continuous vibration signals into discrete time windows. """
    windows = []
    for i in range(0, len(data) - window_size + 1, stride):
        windows.append(data[i:i + window_size])
    return np.array(windows)

def apply_fft(window):
    """ Applies Fast Fourier Transform and discards the 0Hz DC component. """
    fft_values = np.abs(np.fft.fft(window))
    return fft_values[1:513].reshape(512, 1) # Reshaped for Conv1D input

def inject_noise(signal, noise_level):
    """Injects Gaussian noise for robustness testing."""
    if noise_level <= 0: return signal
    noise = np.random.normal(0, 1, signal.shape)
    signal_amplitude = np.max(np.abs(signal))
    return signal + (noise * noise_level * signal_amplitude)

# =============================================================================
# STEP 3: Data Loading & Unsupervised Filtering
# =============================================================================
print("\n=== STEP 3: LOADING IMS DATASET ===")
MAX_SAMPLES_PER_CLASS = 1000

DATASET_PATHS = {
    "IMS": r"/content/drive/MyDrive/Colab Notebooks/FYP/Datasets/IMS/2nd_test"
}

class UniversalDataLoader:
    def __init__(self, name):
        self.name = "IMS"
        self.path = DATASET_PATHS.get("IMS", "")

    def load_data(self, noise_level=0.0):
        print(f"   [{self.name}] Loading Data (Noise {int(noise_level*100)}%)...")
        if not os.path.exists(self.path):
            print(f"Path not found: {self.path}")
            return None, None

        X_data, y_data = [], []
        counts = {0: 0, 1: 0}

        def process_signal(raw_sig, label):
            if counts[label] >= MAX_SAMPLES_PER_CLASS: return
            noisy_sig = inject_noise(raw_sig, noise_level)
            windows = create_sliding_windows(noisy_sig, WINDOW_SIZE, STRIDE)
            for w in windows:
                if counts[label] >= MAX_SAMPLES_PER_CLASS: break
                X_data.append(apply_fft(w))
                y_data.append(label)
                counts[label] += 1

        try:
            if self.name == "IMS":
                files = sorted(os.listdir(self.path))
                DEGRADATION_POINT = 532
                sampled_files = files[::5]

                for f in sampled_files:
                    original_index = files.index(f)
                    label = 0 if original_index < DEGRADATION_POINT else 1
                    try:
                        df = pd.read_csv(os.path.join(self.path, f), sep='\t', header=None)
                        process_signal(df.iloc[:, 0].values, label)
                        if counts[0] >= MAX_SAMPLES_PER_CLASS and counts[1] >= MAX_SAMPLES_PER_CLASS:
                            break
                    except Exception as e:
                        print(f"Error processing {f}: {e}")
            return np.array(X_data), np.array(y_data)
        except Exception as e:
            print(f"Error loading {self.name}: {e}")
            return None, None

loader = UniversalDataLoader("IMS")
X_all, y_all = loader.load_data(noise_level=0.0)

print(f"Data loaded successfully! Total samples: {len(X_all)}")

# Split data first to prevent data leakage
X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=42
)

# 💡 THE AUTOENCODER RULE: ONLY FILTER NORMAL DATA (LABEL 0) FOR TRAINING
X_train_normal = X_train_full[y_train_full == 0]
X_val_normal = X_val_full[y_val_full == 0]

print(f"Training Autoencoder purely on Healthy Data:")
print(f" --> Normal Training samples: {len(X_train_normal)}")
print(f" --> Normal Validation samples: {len(X_val_normal)}")
# =============================================================================
# STEP 4: Constructing the FINAL Lite-Hybrid-AE (Vincent's Architecture)
# =============================================================================
print("\n=== STEP 4: BUILDING LITE-HYBRID-AE MODEL ===")

def build_lite_hybrid_autoencoder(input_shape=(512, 1)):
    inp = layers.Input(shape=input_shape)

    x = layers.Conv1D(filters=16, kernel_size=3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling1D(pool_size=4)(x)

    x = layers.LSTM(32, return_sequences=True, unroll=True)(x)

    encoded = layers.GlobalMaxPooling1D()(x)

    out = layers.Dense(512, activation='linear')(encoded)
    out = layers.Reshape((512, 1))(out)

    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

autoencoder = build_lite_hybrid_autoencoder()
autoencoder.summary()

# =============================================================================
# STEP 5: Model Training (Unsupervised on Normal Data ONLY)
# =============================================================================
print("\n=== STEP 5: INITIATING AUTOENCODER TRAINING ===")

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# 💡 Notice: Input and Target are both X_train_normal (Self-Reconstruction)
history = autoencoder.fit(
    X_train_normal, X_train_normal,
    validation_data=(X_val_normal, X_val_normal),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# =============================================================================
# STEP 6: Edge Deployment Artifact Export (.keras & .tflite)
# =============================================================================
print("\n=== STEP 6: EXPORTING MODELS FOR RASPBERRY PI ===")

# 1. Save standard Keras Model
keras_filename = "Lite_Hybrid_AE_IMS.keras"
autoencoder.save(keras_filename)
print(f" [OK] Standard Model saved: {keras_filename}")

# 2. Convert to pure static TFLite Format
print("Converting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(autoencoder)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_filename = "Lite_Hybrid_AE_Edge.tflite"
with open(tflite_filename, 'wb') as f:
    f.write(tflite_model)
print(f" [OK] TFLite Edge Model saved: {tflite_filename}")

# 3. Trigger Local Downloads
print("\nInitiating downloads to your local machine...")
files.download(keras_filename)
files.download(tflite_filename)